# 📘 MODULE 14.5 — Evaluation — Notes

![Image](https://images.openai.com/static-rsc-4/K8OF149nCcAtRqXK0Y8nMkFoG8pSA6PgncEeRzAxKgoza2akorg19s02ptnMFDSaw6m7v88hE_IdQvZ8zi2W65ZscV-THWTRcNwsV_AX_L7rS8TApcRZPybOhfIpDb8UZ1j66BM75E2FNWYDd6dk1SRFnqpMKpPY_WbbqVCLyqM4z_IV2FyuIFhrwCCZ20Bk?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/9nY6GG9FosrdF_Q__BwwDLfShJf-a8OA0HgDg9oC52qKzp2oEArYBjoZi6UukUceWO7TXxeIKD0Km4SCUy_bGh6Q64_RpCzW2bcNYMwfMMU9jm2btffXL1ik5W5n4nK5iIm2nUlmWQCYOKNVtBFtb93QgIP0CVfV3wZH93cspA5umDhPy85lyELcb3j47lA7?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/kdh0gwhX1N3tjHzZ9NEjrPY3fgGJr4v8HXBOVpma_TqFMBg59vEzUxE13MdQ5NBeKyZw4k-sl3sLu9E7D__Ef58Q1pLaySD5DVE-SwgXTUf3mhChc0bbD0Wlzra8p4W2eTsZbijlKH9jtSLl0jzjPDAzZWNS8wfRVuYTfqTTK5kQEDtCcdiTqEN8zNcbI1wh?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/k-cwINlrM5FNGBAXW38MiB3jtjdX8DaAVckRm3AaqOm0-udMe1unRbhLNsdLyQ3EcPEHhgsRPr83z7Ou2OfbIqy2DGyAizDlj0v8hvZMZ216OiR401A2T1JNN_piOKfMzZQoEb8UJSToL6QhscwymCxh7gQPlGWUkqOe8Itc73rOho_t2G6d3p9m9tp155y9?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/SG6V8YuI6YcYh-0quR6YgTn9z8LCCd4yPOhwIriUcGVV8eRO-j_n3Nyx5LmATgkHUbb3GVnQOt5tTlc_cuSUfBHZB0YXN_QEblqxNtfXUyqaTmOpdXCRNZ0uVVuxfR8oXpY_0KtOF_KAjGVZ7DwZ4wPB2vW1qCjyMOkzhGSEbj7P8vOAVvLBcmQwHL4nNTiu?purpose=fullsize)

> **Core idea:** `model.train()` and `model.eval()` control the **behavior of certain layers**, while `torch.no_grad()` controls **gradient tracking**. They are related to evaluation, but they are **not interchangeable**.

---

# 1. 🔥🔥🔥 `model.train()`

```python
model.train()
```

Puts the model into **training mode**.

This matters especially for layers whose behavior changes between training and evaluation:

* **Dropout**
* **Batch Normalization**

### Dropout

```text id="6uvvuf"
Training mode
     ↓
Dropout ON
     ↓
Some activations randomly dropped
```

### BatchNorm

```text id="j1r5m0"
Training mode
     ↓
Uses current batch statistics
```

So:

> `model.train()` tells the model: **"Use training behavior."**

---

# 2. 🔥🔥🔥 `model.eval()`

```python
model.eval()
```

Puts the model into **evaluation mode**.

### Dropout

```text id="3z0r6r"
Evaluation mode
      ↓
Dropout OFF
```

### BatchNorm

```text id="qz4v1j"
Evaluation mode
      ↓
Uses stored running statistics
```

So:

> `model.eval()` tells the model: **"Use evaluation behavior."**

---

# 3. 🔥🔥🔥 `torch.no_grad()`

```python
with torch.no_grad():
    prediction = model(X)
```

This tells PyTorch:

> **Do not track gradients for these operations.**

This is useful during:

* validation
* testing
* inference

because we aren't updating model parameters.

---

# 4. 🚨 `eval()` ≠ `no_grad()`

This is the **most important concept in this block**.

|                    | `model.train()` | `model.eval()` | `torch.no_grad()`  |
| ------------------ | --------------- | -------------- | ------------------ |
| Changes model mode | ✅ Training      | ✅ Evaluation   | ❌                  |
| Dropout behavior   | Training        | Evaluation     | Doesn't control it |
| BatchNorm behavior | Training        | Evaluation     | Doesn't control it |
| Disables gradients | ❌               | ❌              | ✅                  |
| Freezes parameters | ❌               | ❌              | ❌                  |

### 🧠 Memory trick

> **`train()` / `eval()` → control layer behavior**

> **`no_grad()` → controls gradient tracking**

---

# 5. 🔥 Correct Evaluation Pattern

The usual validation/test/inference pattern is:

```python
model.eval()

with torch.no_grad():
    prediction = model(X)
```

Why both?

```text id="v4v3m5"
model.eval()
      ↓
Correct evaluation behavior
      +
torch.no_grad()
      ↓
No gradient tracking
```

They solve **different problems**.

---

# 6. 🔥 Training vs Evaluation

## Training

```python
model.train()

for X_batch, y_batch in train_loader:

    prediction = model(X_batch)

    loss = loss_fn(prediction, y_batch)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()
```

Flow:

```text id="2om7u7"
Data
 ↓
Training mode
 ↓
Prediction
 ↓
Loss
 ↓
Backward
 ↓
Gradients
 ↓
Optimizer
 ↓
Update
```

---

## Evaluation

```python
model.eval()

with torch.no_grad():

    for X_batch, y_batch in val_loader:

        prediction = model(X_batch)

        loss = loss_fn(prediction, y_batch)
```

Flow:

```text id="y1n1dr"
Data
 ↓
Evaluation mode
 ↓
Prediction
 ↓
Loss
 ↓
No backward
 ↓
No parameter update
```

---

# 7. 🔥 Why No `backward()` During Evaluation?

During training, we want the model to **learn**.

```text id="yfr0c9"
Prediction
 ↓
Loss
 ↓
Gradients
 ↓
Update
```

During evaluation, we only want to **measure performance**.

```text id="1h0l6f"
Prediction
 ↓
Loss / Metric
 ↓
Measure
```

We don't want validation/test data to update the model.

---

# 8. 🔥 `eval()` Does NOT Freeze Parameters

This is a common mistake.

```python
model.eval()
```

does **not** mean:

> "Freeze every parameter."

It does not automatically:

* set `requires_grad=False`
* disable gradient computation
* freeze parameters
* prevent optimizer updates

It changes the behavior of modules such as Dropout and BatchNorm.

---

## Freezing is a separate concept

To freeze parameters:

```python
for param in model.parameters():
    param.requires_grad = False
```

This is different from:

```python
model.eval()
```

### Remember:

```text id="jtrq2g"
model.eval()
      ↓
Evaluation behavior

requires_grad=False
      ↓
Parameter gradient tracking disabled
```

These are different concepts.

---

# 9. 🔥 `no_grad()` Does NOT Set Evaluation Mode

This is the reverse misconception.

Suppose:

```python
model.train()

with torch.no_grad():
    prediction = model(X)
```

`no_grad()` disables gradient tracking, **but the model is still in training mode**.

Therefore:

* Dropout can still behave as training Dropout.
* BatchNorm can still use training behavior.

So for normal evaluation:

```python
model.eval()

with torch.no_grad():
    prediction = model(X)
```

---

# 10. 🔥 Complete Training + Validation Structure

```python
for epoch in range(num_epochs):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    for X_batch, y_batch in train_loader:

        prediction = model(X_batch)

        loss = loss_fn(prediction, y_batch)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            prediction = model(X_batch)

            loss = loss_fn(prediction, y_batch)
```

The important transition is:

```text id="x1t6cz"
model.train()
      ↓
Training batches
      ↓
model.eval()
      ↓
Validation batches
```

---

# 11. 🔥 Three Concepts Side-by-Side

### `model.train()`

> **"Use training behavior."**

```python
model.train()
```

### `model.eval()`

> **"Use evaluation behavior."**

```python
model.eval()
```

### `torch.no_grad()`

> **"Don't track gradients here."**

```python
with torch.no_grad():
    ...
```

---

# 🧠 Complete Mental Model

```text id="7b0z9a"
                     MODEL
                       │
          ┌────────────┴────────────┐
          │                         │
     model.train()             model.eval()
          │                         │
          ▼                         ▼
 Training behavior           Evaluation behavior
          │                         │
     Dropout ON               Dropout OFF
     BatchNorm               BatchNorm uses
     training stats           running stats


              torch.no_grad()
                     │
                     ▼
             Gradient tracking
                  OFF
```

The key is that these are **two independent controls**:

```text
train()/eval()
     ↓
Model behavior

no_grad()
     ↓
Gradient tracking
```

---

# 🎯 Quick Revision Table

| Situation  | Recommended setup                  |
| ---------- | ---------------------------------- |
| Training   | `model.train()`                    |
| Validation | `model.eval()` + `torch.no_grad()` |
| Test       | `model.eval()` + `torch.no_grad()` |
| Inference  | `model.eval()` + `torch.no_grad()` |

---

# 🔥 Final Cheat Sheet

```text id="o7a0m8"
TRAINING
────────────────────
model.train()

Forward
   ↓
Loss
   ↓
Backward
   ↓
Optimizer
   ↓
Update


VALIDATION / TEST / INFERENCE
─────────────────────────────
model.eval()

with torch.no_grad():

    Forward
       ↓
    Prediction
       ↓
    Loss / Metric
```

### 🧠 One sentence to remember

> **`model.train()` and `model.eval()` switch the model between training and evaluation behavior, while `torch.no_grad()` prevents Autograd from tracking gradients; therefore evaluation normally uses both `model.eval()` and `torch.no_grad()`.**

---